# Used Car Price Prediction

Predicting a used car's selling price from its mileage, brand, engine size, fuel type, transmission, and other features using regression models.

**Pipeline:** Data Cleaning → EDA → Feature Engineering → Regression Modeling → Evaluation → Model Comparison

## 1. Load Data

In [3]:
import numpy as np
import pandas as pd

df = pd.read_csv('used_car_price_data.csv')
df.shape

(4012, 12)

## 2. Data Cleaning

### 2.1 Missing Values

In [4]:
df.isnull().sum()

car_id               0
brand                0
car_age_years        0
mileage_km          80
engine_cc           60
fuel_type            0
transmission         0
owner_count         40
accident_history     0
seats                0
city                 0
selling_price        0
dtype: int64

In [5]:
# Drop rows with missing values (small % of data, no reliable way to impute a fair guess for these columns)
df = df.dropna()
df.shape

(3836, 12)

### 2.2 Duplicates

In [6]:
df.duplicated().sum()

np.int64(12)

In [7]:
df = df.drop_duplicates()
df.shape

(3824, 12)

### 2.3 Standardize Column Names

In [8]:
df.columns = df.columns.str.strip().str.lower()

### 2.4 Clean Messy Text (`brand`)
Raw data has inconsistent capitalization and typos (e.g. `TOYOTA`, `toyota`, `Toyoota`).

In [9]:
df['brand'].unique()

<StringArray>
[     'TOYOTA',      'Toyota',  'Mitsubishi',      'toyota',       'Honda',
       'HONDA',      'nissan',      'Suzuki',  'mitsubishi',      'NISSAN',
     'hyundai',      'SUZUKI',       'honda',     'Hyundai',      'Nissan',
     'Toyoota',      'Hyndai', 'Mitsubishi ',      'suzuki']
Length: 19, dtype: str

In [10]:
df['brand'] = df['brand'].str.strip().str.lower()
df['brand'] = df['brand'].replace({
    'toyoota': 'toyota',
    'hyndai': 'hyundai'
})
df['brand'] = df['brand'].str.title()
df['brand'].unique()

<StringArray>
['Toyota', 'Mitsubishi', 'Honda', 'Nissan', 'Suzuki', 'Hyundai']
Length: 6, dtype: str

### 2.5 Sanity Check

In [11]:
df.describe()

,car_id,car_age_years,mileage_km,engine_cc,owner_count,accident_history,seats,selling_price
count,3824.000000,3824.000000,3824.000000,3824.000000,3824.000000,3824.000000,3824.000000,3.824000e+03
mean,2004.949529,7.466789,89866.486402,1390.899582,1.881538,0.201621,5.172071,1.421109e+06
std,1154.260927,4.614679,60872.910191,288.312446,0.972730,0.401263,0.810405,6.923279e+05
min,1.000000,0.000000,500.000000,1000.000000,1.000000,0.000000,4.000000,1.500000e+05
25%,1006.750000,3.000000,38772.000000,1200.000000,1.000000,0.000000,5.000000,8.940000e+05
50%,2013.000000,7.000000,84198.000000,1300.000000,2.000000,0.000000,5.000000,1.413000e+06
75%,3000.250000,12.000000,134206.750000,1500.000000,3.000000,0.000000,5.000000,1.956250e+06
max,4000.000000,15.000000,250000.000000,2000.000000,4.000000,1.000000,7.000000,3.108000e+06


## 3. Exploratory Data Analysis

### 3.1 Multicollinearity Check
`car_age_years` and `mileage_km` both describe vehicle wear — check if they overlap too much.

In [12]:
df['car_age_years'].corr(df['mileage_km'])

np.float64(0.9009242496912755)

**Finding:** 0.90 correlation — strongly overlapping. Both carry nearly the same information.

### 3.2 Relationship with Price

In [13]:
print('mileage_km vs price:', df['mileage_km'].corr(df['selling_price']))
print('car_age_years vs price:', df['car_age_years'].corr(df['selling_price']))

mileage_km vs price: -0.8049831926825397
car_age_years vs price: -0.8393461006644048


**Finding:** Both strongly negatively correlated with price (~-0.80 and ~-0.84). Since they're also highly correlated with each other, keeping both would cause multicollinearity in the regression model.

**Decision:** Keep `mileage_km`, drop `car_age_years` — mileage is a more direct indicator of actual wear (a car can be old but lightly used, or young but heavily driven).

In [14]:
df = df.drop(columns='car_age_years')

### 3.3 Categorical Relationships with Price

In [15]:
df.groupby('transmission')['selling_price'].mean()

transmission
Automatic    1.479010e+06
Manual       1.350132e+06
Name: selling_price, dtype: float64

In [16]:
df.groupby('fuel_type')['selling_price'].mean()

fuel_type
CNG       1.343106e+06
Diesel    1.428815e+06
Hybrid    1.530273e+06
Petrol    1.392858e+06
Name: selling_price, dtype: float64

**Findings:**
- Automatic transmission cars sell for more on average than Manual
- Hybrid > Diesel > Petrol > CNG in average price — matches real-world expectations

## 4. Feature Engineering

### 4.1 One-Hot Encoding

In [17]:
df.columns.tolist()

['car_id',
 'brand',
 'mileage_km',
 'engine_cc',
 'fuel_type',
 'transmission',
 'owner_count',
 'accident_history',
 'seats',
 'city',
 'selling_price']

In [18]:
df = pd.get_dummies(df, columns=['brand', 'fuel_type', 'transmission', 'city'])
df.shape

(3824, 24)

### 4.2 Features (X) / Target (y) Split

In [19]:
y = df['selling_price']
x = df.drop(columns=['selling_price', 'car_id'])  # car_id is just a label, not predictive
x.shape, y.shape

((3824, 22), (3824,))

### 4.3 Train/Test Split

In [20]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.shape, x_test.shape

((3059, 22), (765, 22))

### 4.4 Feature Scaling
Regression models can be sensitive to features on very different scales (e.g. `mileage_km` in the hundred-thousands vs `owner_count` from 1-4). `StandardScaler` puts every feature on a comparable scale.

**Important:** fit the scaler on training data only, then apply (transform) the same scaling rule to test data — never fit on test data, to avoid leaking test information into training.

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler_x_train = scaler.fit_transform(x_train)
scaler_x_test = scaler.transform(x_test)

## 5. Modeling

### 5.1 Linear Regression

In [22]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(scaler_x_train, y_train)
lr_pred = lr_model.predict(scaler_x_test)

### 5.2 Decision Tree Regressor

In [23]:
from sklearn.tree import DecisionTreeRegressor

tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(scaler_x_train, y_train)
tree_pred = tree_model.predict(scaler_x_test)

### 5.3 Random Forest Regressor

In [24]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(scaler_x_train, y_train)
rf_pred = rf_model.predict(scaler_x_test)

## 6. Evaluation

Regression models are evaluated differently from classifiers — instead of accuracy/recall/precision, we measure how far predictions are from actual values:
- **MAE** (Mean Absolute Error): average error size, in real currency, treating all errors equally
- **RMSE** (Root Mean Squared Error): like MAE, but penalizes large errors much more heavily
- **R²**: the % of price variation explained by the model (1.0 = perfect, 0 = no better than guessing the average)

In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name}: MAE={mae:,.0f}  RMSE={rmse:,.0f}  R2={r2:.3f}')
    return mae, rmse, r2

results = {}
results['Linear Regression'] = evaluate('Linear Regression', y_test, lr_pred)
results['Decision Tree'] = evaluate('Decision Tree', y_test, tree_pred)
results['Random Forest'] = evaluate('Random Forest', y_test, rf_pred)

Linear Regression: MAE=163,555  RMSE=218,615  R2=0.899
Decision Tree: MAE=231,927  RMSE=310,943  R2=0.795
Random Forest: MAE=162,502  RMSE=213,071  R2=0.904


### Model Comparison

| Model | MAE | RMSE | R² |
|---|---|---|---|
| Linear Regression | 163,555 | 218,615 | 0.899 |
| Decision Tree | 231,927 | 310,943 | 0.795 |
| **Random Forest** | **162,502** | **213,071** | **0.904** |

**Random Forest performed best on all three metrics.** The single Decision Tree performed worst — likely overfitting since it was allowed to grow to unlimited depth, memorizing noise in the training data rather than learning general patterns.

### Feature Importance (Random Forest)

In [26]:
importances = pd.Series(rf_model.feature_importances_, index=x.columns).sort_values(ascending=False)
importances.head(10)

mileage_km          0.711837
brand_Toyota        0.096245
brand_Honda         0.066446
brand_Suzuki        0.034062
engine_cc           0.020305
owner_count         0.014465
accident_history    0.010425
seats               0.006814
fuel_type_Hybrid    0.004985
fuel_type_Petrol    0.004070
dtype: float64

**Finding:** `mileage_km` dominates (~71% of importance) — confirming the strong correlation found during EDA. Brand (especially Toyota, Honda) is the next most influential factor, matching real-world market perception of resale value.

## 7. Conclusion

A Random Forest Regressor was able to explain ~90% of the variation in used car selling prices (R² = 0.904), with an average prediction error of about 162,500 taka. Mileage is by far the strongest driver of price, followed by brand. This project also surfaced and resolved a real multicollinearity issue (age vs. mileage) before modeling, and demonstrated that an unconstrained Decision Tree can overfit and underperform compared to both a simpler linear model and an ensemble method.

**Possible future improvements:**
- Hyperparameter tuning (`GridSearchCV`) for Decision Tree / Random Forest
- Try Gradient Boosting / XGBoost
- Investigate and cap the outlier predictions that widen the MAE/RMSE gap